# Étape 6 — Manipulations : l'inférence RDFS en action

Le principe de ce notebook : **avant / après**. On lance les mêmes requêtes deux fois — une fois sur le graphe brut, une fois après avoir activé le raisonneur — pour voir apparaître les connaissances déduites.

> Prérequis : `pip install -r requirements.txt` (inclut `owlrl`). Exécute les cellules dans l'ordre.

In [1]:
from rdflib import Graph

graphe = Graph()
graphe.parse("films.ttl", format="turtle")
print(f"Triplets AVANT inference : {len(graphe)}\n")


def afficher(requete: str) -> None:
    resultats = graphe.query(requete)
    if not resultats:
        print("  (aucun resultat)")
        return
    for ligne in resultats:
        valeurs = {str(var): str(val) for var, val in ligne.asdict().items()}
        print(" ", valeurs)


# Les requetes qu'on va relancer avant ET apres inference :
REQ_PERSONNES = """
PREFIX ex: <http://exemple.org/cinema/>
SELECT ?nom WHERE {
    ?p a ex:Personne .
    ?p ex:nom ?nom .
}
"""

REQ_FILMS = """
PREFIX ex: <http://exemple.org/cinema/>
SELECT ?f WHERE { ?f a ex:Film . }
"""

Triplets AVANT inference : 44



## 1. AVANT : le graphe brut

On demande les Personnes, puis les Films. Sans raisonneur :
- **Personnes** : aucun résultat (personne n'est typé `Personne` explicitement).
- **Films** : seulement `inception` et `granTorino` (pas `dune`, qui n'est pas typé).

In [2]:
print("--- Personnes (avant) ---")
afficher(REQ_PERSONNES)

print("\n--- Films (avant) ---")
afficher(REQ_FILMS)

--- Personnes (avant) ---
  (aucun resultat)

--- Films (avant) ---
  {'f': 'http://exemple.org/cinema/inception'}
  {'f': 'http://exemple.org/cinema/granTorino'}


## 2. On lance le raisonneur RDFS

`owlrl` calcule la clôture RDFS et **ajoute** les triplets déduits directement dans le graphe. On compare le nombre de triplets avant/après.

In [3]:
import owlrl

avant = len(graphe)

# RDFS_Semantics applique les regles RDFS (subClassOf, domain, range, ...).
owlrl.RDFSClosure.RDFS_Semantics(graphe, axioms=False, daxioms=False, rdfs=True).closure()

apres = len(graphe)
print(f"Triplets avant : {avant}")
print(f"Triplets apres : {apres}")
print(f"Triplets deduits par le raisonneur : {apres - avant}")

Triplets avant : 44
Triplets apres : 114
Triplets deduits par le raisonneur : 70


## 3. APRÈS : les mêmes requêtes

On relance **exactement** les mêmes requêtes. Maintenant :
- **Personnes** : Nolan, DiCaprio, Eastwood... et même Villeneuve (déduit via domain + subClassOf) !
- **Films** : `dune` apparaît aussi (déduit via le range de `aRealise`).

In [4]:
print("--- Personnes (apres) ---")
afficher(REQ_PERSONNES)

print("\n--- Films (apres) ---")
afficher(REQ_FILMS)

--- Personnes (apres) ---
  {'nom': 'Christopher Nolan'}
  {'nom': 'Clint Eastwood'}
  {'nom': 'Denis Villeneuve'}
  {'nom': 'Leonardo DiCaprio'}

--- Films (apres) ---
  {'f': 'http://exemple.org/cinema/inception'}
  {'f': 'http://exemple.org/cinema/granTorino'}
  {'f': 'http://exemple.org/cinema/dune'}


## 4. Zoom : qu'a-t-on déduit pour Villeneuve ?

On affiche tous les types (`rdf:type`) déduits pour `ex:villeneuve`. On n'avait écrit aucun type dans le fichier, pourtant le raisonneur lui en attribue plusieurs.

In [5]:
afficher("""
PREFIX ex:  <http://exemple.org/cinema/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?type
WHERE {
    ex:villeneuve a ?type .
    ?type rdfs:label ?label .   # on ne garde que NOS classes (celles qui ont un label)
}
""")

  {'type': 'http://exemple.org/cinema/Personne'}
  {'type': 'http://exemple.org/cinema/Realisateur'}


## À toi de jouer

**Exercice :** après inférence, écris une requête qui liste **tous les Acteurs**. Vérifie que DiCaprio et Eastwood y sont. (Eastwood est explicitement Acteur ; le but est surtout de constater que la requête sur une classe précise marche toujours.)

Bonus de réflexion : pourquoi Villeneuve n'apparaît-il PAS dans les Acteurs ? (Réponse : on a seulement dit qu'il *a réalisé* un film, jamais qu'il a *joué* — rien ne permet de déduire qu'il est Acteur.)

In [6]:
# TODO : complete cette requete pour lister les Acteurs
afficher("""
PREFIX ex: <http://exemple.org/cinema/>

SELECT ?nom
WHERE {
    ?p ex:nom ?nom .
    # ... a completer ...
}
""")

  {'nom': 'Christopher Nolan'}
  {'nom': 'Leonardo DiCaprio'}
  {'nom': 'Clint Eastwood'}
  {'nom': 'Denis Villeneuve'}


<details>
<summary>Afficher la solution</summary>

```python
afficher("""
PREFIX ex: <http://exemple.org/cinema/>

SELECT ?nom
WHERE {
    ?p a ex:Acteur .
    ?p ex:nom ?nom .
}
""")
```

Résultat : Leonardo DiCaprio, Clint Eastwood.

</details>

---

Tu as vu la puissance de l'inférence RDFS. Mais RDFS ne sait pas tout exprimer (relations symétriques, inverses, transitives...). C'est le rôle d'**OWL**, à l'**étape 7**.